In [1]:
# Se define la ruta del conjunto de datos del caso.

DATA = "../data/auto_mpg.csv"
OUTPUT = "../submission"
ESTIMATOR = f"{OUTPUT}/estimator.pkl"

In [2]:
# Estos contratos detienen el flujo antes de entrenar o sobrescribir artefactos con insumos inválidos.

import os
from pathlib import Path

DATA_PATH = Path(DATA)
OUTPUT_PATH = Path(OUTPUT)
assert DATA_PATH.is_file(), f"No se encontró el archivo de datos: {DATA_PATH}"
assert DATA_PATH.stat().st_size > 0, f"El archivo de datos está vacío: {DATA_PATH}"
assert OUTPUT_PATH.is_dir(), f"No se encontró el directorio de artefactos: {OUTPUT_PATH}"
assert os.access(OUTPUT_PATH, os.W_OK), f"No hay permiso de escritura en: {OUTPUT_PATH}"

In [3]:
# Se define la carga de los datos que alimentan el taller.

def load_data():

    import pandas as pd

    dataset = pd.read_csv(DATA)
    dataset = dataset.dropna()
    dataset["Origin"] = dataset["Origin"].map(
        {1: "USA", 2: "Europe", 3: "Japan"},
    )
    y = dataset.pop("MPG")
    x = dataset.copy()

    return x, y

In [4]:
# Se define la separación entre entrenamiento y prueba.

def make_train_test_split(x, y):

    from sklearn.model_selection import train_test_split

    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.25,
        random_state=123456,
    )
    return x_train, x_test, y_train, y_test

In [5]:
# Se define el pipeline para conservar las transformaciones junto al modelo.

def make_pipeline(estimator):

    from sklearn.compose import ColumnTransformer
    from sklearn.feature_selection import SelectKBest, f_regression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    transformer = ColumnTransformer(
        transformers=[
            ("ohe", OneHotEncoder(dtype="int"), ["Origin"]),
        ],
        remainder=StandardScaler(),
    )

    selectkbest = SelectKBest(score_func=f_regression)

    pipeline = Pipeline(
        steps=[
            ("tranformer", transformer),
            ("selectkbest", selectkbest),
            ("estimator", estimator),
        ],
        verbose=False,
    )

    return pipeline

In [6]:
# Se define la búsqueda de parámetros mediante validación cruzada.

def make_grid_search(estimator, param_grid, cv=5):

    from sklearn.model_selection import GridSearchCV

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="neg_mean_absolute_error",
    )

    return grid_search

In [7]:
# Se define la carga del estimador almacenado.

def load_estimator():

    import os
    import pickle

    if not os.path.exists(ESTIMATOR):
        return None
    with open(ESTIMATOR, "rb") as file:
        estimator = pickle.load(file)

    return estimator

In [8]:
# Se define el almacenamiento del estimador para reutilizarlo posteriormente.

def save_estimator(estimator, x, y):

    import os
    import pickle

    from sklearn.metrics import mean_absolute_error

    best_estimator = load_estimator()

    if best_estimator is not None:

        saved_mae = mean_absolute_error(y_true=y, y_pred=best_estimator.predict(x))

        current_mae = mean_absolute_error(y_true=y, y_pred=estimator.predict(x))

        if saved_mae < current_mae:
            estimator = best_estimator

    with open(ESTIMATOR, "wb") as file:
        pickle.dump(estimator, file)

In [9]:
# Se define el entrenamiento del modelo lineal de referencia.

def train_linear_regression():

    import os

    from sklearn.linear_model import LinearRegression

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    pipeline = make_pipeline(
        estimator=LinearRegression(),
    )

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid={
            "selectkbest__k": range(1, len(x_train.columns) + 1),
        },
        cv=5,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)


train_linear_regression()

In [10]:
# Se definen las métricas para comparar los modelos.

def eval_metrics(y_true, y_pred):

    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, mae, r2

In [11]:
# Se define el reporte de métricas para discutir la comparación.

def report(estimator, mse, mae, r2):

    print(estimator, ":", sep="")
    print(f"  MSE: {mse}")
    print(f"  MAE: {mae}")
    print(f"   R2: {r2}")

In [12]:
# Se define la verificación del estimador que quedó almacenado.

def check_estimator():

    data, target = load_data()

    _, x_test, _, y_test_true = make_train_test_split(
        x=data,
        y=target,
    )

    estimator = load_estimator()

    mse, mae, r2 = eval_metrics(
        y_test_true,
        estimator.predict(x_test),
    )

    report(estimator.best_estimator_, mse, mae, r2)


check_estimator()

Pipeline(steps=[('tranformer',
                 ColumnTransformer(remainder=StandardScaler(),
                                   transformers=[('ohe',
                                                  OneHotEncoder(dtype='int'),
                                                  ['Origin'])])),
                ('selectkbest',
                 SelectKBest(k=6,
                             score_func=<function f_regression at 0x1568d28b0>)),
                ('estimator', LinearRegression())]):
  MSE: 11.177256954645403
  MAE: 2.524517976430189
   R2: 0.8089190863428402


In [13]:
# Se define el entrenamiento del regresor neuronal para compararlo.

def train_mlp_regressor():

    from sklearn.neural_network import MLPRegressor

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    pipeline = make_pipeline(
        estimator=MLPRegressor(max_iter=1000),
    )

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid={
            "selectkbest__k": range(1, 5),
            "estimator__hidden_layer_sizes": [(n,) for n in range(1, 3)],
            "estimator__solver": ["adam"],
            "estimator__learning_rate_init": [0.01, 0.001, 0.0001],
        },
        cv=5,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)


# train_mlp_regressor()
# check_estimator()